In [1]:
import tensorflow as tf
from tensorflow import keras
from keras import layers,models
import matplotlib.pyplot as plt
import numpy as np

In [2]:
import os
from google.colab import drive
drive.mount('/content/drive/')
data_dir="/content/drive/MyDrive/animals"

Mounted at /content/drive/


In [3]:
img_size=(128,128)
batch_size=32

training_data=tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

Found 5406 files belonging to 90 classes.
Using 4325 files for training.


In [4]:
validation_data=tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

Found 5406 files belonging to 90 classes.
Using 1081 files for validation.


In [5]:
class_names=training_data.class_names
num_classes=len(class_names)
print(num_classes)

90


In [6]:
augment=tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

In [7]:
AUTOTUNE=tf.data.AUTOTUNE

training_data=(training_data.map(lambda x,y:(augment(x,training=True)/255.0,y)).prefetch(AUTOTUNE))
validation_data=(validation_data.map(lambda x,y:(x/255.0,y)).prefetch(AUTOTUNE))

In [8]:
# Build a Custom ResNet
def residual_block(x,filters,stride=1):
  shortcut=x

  x=layers.Conv2D(filters,3,strides=stride,padding='same',use_bias=False)(x)
  x=layers.BatchNormalization()(x)
  x=layers.ReLU()(x)

  x=layers.Conv2D(filters,3,padding="same",use_bias=False)(x)
  x=layers.BatchNormalization()(x)


  if stride!=1 or shortcut.shape[-1]!=filters:
    shortcut=layers.Conv2D(filters,1,strides=stride,use_bias=False)(shortcut)
  x=layers.Add()([x,shortcut]) #skip connection or identity path
  x=layers.ReLU()(x)

  return x

In [9]:
# Build the Full Custom ResNet

def build_resnet(num_classes,input_shape=(128,128,3)):

  inputs=layers.Input(shape=input_shape)

  x=layers.Conv2D(64,7,strides=2,padding="same",use_bias=False)(inputs)
  x=layers.BatchNormalization()(x)
  x=layers.ReLU()(x)
  x=layers.MaxPooling2D(3,strides=2,padding="same")(x)

  x=residual_block(x,64)
  x=residual_block(x,64)


  x=residual_block(x,128,stride=2)
  x=residual_block(x,128)

  x=residual_block(x,256,stride=2)
  x=residual_block(x,256)

  x=layers.GlobalAveragePooling2D()(x)
  x=layers.Dense(256,activation='relu')(x)
  x=layers.Dropout(0.4)(x)
  outputs=layers.Dense(num_classes,activation="softmax")(x)

  model=tf.keras.Model(inputs,outputs)
  return model


model=build_resnet(num_classes)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 64,    │      9,408 │ input_layer_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 64,    │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 32, 32,    │          0 │ re_lu[0][0]       │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │     36,864 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     36,864 │ re_lu_1[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ max_pooling2d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 32, 32,    │          0 │ add[0][0]         │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │     36,864 │ re_lu_2[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │     36,864 │ re_lu_3[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_4[0][0]  

 Total params: 2,874,650 (10.97 MB)

 Trainable params: 2,870,938 (10.95 MB)

 Non-trainable params: 3,712 (14.50 KB)

In [10]:
# Learning Rate Schedule

lr_schedule=tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=50*len(training_data)
)

In [11]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(lr_schedule),
    loss="sparse_categorical_crossentropy",
    metrics=['accuracy']
)

In [12]:
callbacks=[
    tf.keras.callbacks.EarlyStopping(patience=9,restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("best_wildlife_resnet.keras",save_best_only=True)

]

In [ ]:
history=model.fit(
    training_data,
    validation_data=validation_data,
    epochs=50,
    callbacks=callbacks
)

Epoch 1/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 1173s 9s/step - accuracy: 0.0291 - loss: 4.4878 - val_accuracy: 0.0167 - val_loss: 4.6462
Epoch 2/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 47s 344ms/step - accuracy: 0.0391 - loss: 4.3173 - val_accuracy: 0.0296 - val_loss: 4.2950
Epoch 3/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 46s 340ms/step - accuracy: 0.0490 - loss: 4.1793 - val_accuracy: 0.0407 - val_loss: 4.2732
Epoch 4/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 46s 336ms/step - accuracy: 0.0564 - loss: 4.1345 - val_accuracy: 0.0176 - val_loss: 8.8883
Epoch 5/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 46s 341ms/step - accuracy: 0.0608 - loss: 4.0379 - val_accuracy: 0.0500 - val_loss: 4.2561
Epoch 6/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 46s 338ms/step - accuracy: 0.0682 - loss: 3.9797 - val_accuracy: 0.0629 - val_loss: 4.1150
Epoch 7/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 50s 370ms/step - accuracy: 0.0809 - loss: 3.9158 - val_accuracy: 0.0398 - val_loss: 4.7001
Epoch 8/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 48s 352ms/step - accuracy: 0.0925 - loss: 3.

In [ ]:

# Plot Accuracy/Loss

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(["Training","Validation"])
plt.title("Accuracy Graph")
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.legend(["Training","Validation"])
plt.title("Loss Graph")
plt.show()